# Notebook 06 — Error Analysis: Where Does the Baseline Fail?

The traditional peak-detection baseline missed **1,686 axles** (11.6 % recall gap) on the test set.  
This notebook investigates *why*: which vehicles are affected, and what signal characteristics correlate with failure.

Questions answered:
1. How are failed vehicles distributed across axle-count categories?
2. Is axle spacing the driver — do closely-spaced axles confuse the baseline?
3. What do the signal waveforms look like for missed vs detected axles?


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import Counter, defaultdict

from src.dataset import build_datasets
from src.baseline import signal_to_pulse_peaks
from src.evaluate import axle_level_metrics, pulses_to_peaks, _match_peaks

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

JSON_PATH = '../axle_data.json/axle_data.json'
SEED      = 42
TOL       = 5

_, _, test_ds = build_datasets(JSON_PATH, seed=SEED)

test_signals = [test_ds[i][0].squeeze(0).numpy() for i in range(len(test_ds))]
test_pulses  = [test_ds[i][1].numpy()             for i in range(len(test_ds))]

print(f'Test set: {len(test_ds):,} vehicles')


In [ ]:
# Run baseline on all test vehicles, record per-vehicle FN counts
pred_base = [signal_to_pulse_peaks(s) for s in test_signals]

per_vehicle = []   # dict per vehicle: axle_count, fn, tp, fp, min_spacing, mean_spacing
for sig, pulse, pred in zip(test_signals, test_pulses, pred_base):
    true_peaks = pulses_to_peaks(pulse, threshold=0.5)
    tp, fp, fn = _match_peaks(pred, true_peaks, tolerance=TOL)
    n_axles = len(true_peaks)

    # Axle spacing statistics from ground truth
    spacings = np.diff(np.sort(true_peaks)).tolist() if len(true_peaks) > 1 else []
    min_sp   = min(spacings) if spacings else np.nan
    mean_sp  = float(np.mean(spacings)) if spacings else np.nan

    per_vehicle.append({
        'n_axles':     n_axles,
        'tp':          tp,
        'fp':          fp,
        'fn':          fn,
        'min_spacing': min_sp,
        'mean_spacing': mean_sp,
        'has_fn':      fn > 0,
    })

import pandas as pd
pv = pd.DataFrame(per_vehicle)
print(f"Vehicles with ≥1 missed axle : {pv['has_fn'].sum():,}  "
      f"({100*pv['has_fn'].mean():.1f} % of test set)")
print(f"Total missed axles (FN)      : {int(pv['fn'].sum()):,}")


## 1. Failure Rate by Axle Count


In [ ]:
axle_counts = sorted(pv['n_axles'].unique())
fail_rates  = []
vehicle_counts = []

for n in axle_counts:
    subset = pv[pv['n_axles'] == n]
    fail_rates.append(100 * subset['has_fn'].mean())
    vehicle_counts.append(len(subset))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: proportion of vehicles with ≥1 FN per axle count
color_fail = ['#e74c3c' if r > 0 else '#2ecc71' for r in fail_rates]
bars = axes[0].bar(axle_counts, fail_rates, color=color_fail, edgecolor='white')
for bar, r in zip(bars, fail_rates):
    if r > 0:
        axes[0].text(bar.get_x() + bar.get_width()/2, r + 0.5,
                     f'{r:.1f}%', ha='center', va='bottom', fontsize=8)
axes[0].set_xlabel('Number of axles on vehicle')
axes[0].set_ylabel('% of vehicles with ≥1 missed axle')
axes[0].set_title('Baseline Failure Rate by Axle Count')
axes[0].set_xticks(axle_counts)

# Right: vehicle count distribution (context)
axes[1].bar(axle_counts, vehicle_counts, color='steelblue', edgecolor='white')
axes[1].set_xlabel('Number of axles on vehicle')
axes[1].set_ylabel('Vehicle count (test set)')
axes[1].set_title('Test Set — Axle Count Distribution')
axes[1].set_xticks(axle_counts)

plt.tight_layout()
plt.show()


## 2. Axle Spacing Distribution: Failed vs Successful Vehicles


In [ ]:
failed_spacings  = pv[pv['has_fn']]['min_spacing'].dropna().values
success_spacings = pv[~pv['has_fn']]['min_spacing'].dropna().values

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histogram of minimum inter-axle spacing
bins = np.arange(0, 200, 5)
axes[0].hist(failed_spacings,  bins=bins, alpha=0.7, color='#e74c3c',  label=f'Failed  (n={len(failed_spacings):,})')
axes[0].hist(success_spacings, bins=bins, alpha=0.6, color='steelblue', label=f'Success (n={len(success_spacings):,})')
axes[0].set_xlabel('Minimum inter-axle spacing (samples)')
axes[0].set_ylabel('Vehicle count')
axes[0].set_title('Minimum Axle Spacing Distribution')
axes[0].legend()
axes[0].set_xlim(0, 200)

# Box plot: minimum spacing by outcome
data_bp   = [failed_spacings, success_spacings]
bp = axes[1].boxplot(data_bp, labels=['Failed', 'Success'], patch_artist=True,
                      medianprops={'color': 'black', 'linewidth': 2})
bp['boxes'][0].set_facecolor('#e74c3c')
bp['boxes'][1].set_facecolor('steelblue')
for patch in bp['boxes']:
    patch.set_alpha(0.7)
axes[1].set_ylabel('Minimum axle spacing (samples)')
axes[1].set_title('Minimum Spacing: Failed vs Successful')

# Annotate medians
for i, d in enumerate(data_bp, 1):
    axes[1].text(i, np.median(d) + 1, f'med={np.median(d):.0f}',
                 ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"Median min-spacing — Failed:  {np.median(failed_spacings):.1f} samples")
print(f"Median min-spacing — Success: {np.median(success_spacings):.1f} samples")


## 3. Example Waveforms — Baseline Failures

Showing 6 vehicles where the baseline missed at least one axle. Red vertical lines = ground-truth axle positions. Blue markers = baseline predictions. Orange triangles = missed axles (FN).


In [ ]:
failed_indices = pv.index[pv['has_fn']].tolist()

# Sort failed vehicles by fn count descending for most illustrative examples
failed_fn_sorted = sorted(failed_indices, key=lambda i: pv.loc[i, 'fn'], reverse=True)
show_n = min(6, len(failed_fn_sorted))
example_indices = failed_fn_sorted[:show_n]

fig, axes = plt.subplots(show_n, 1, figsize=(14, 2.8 * show_n))
if show_n == 1:
    axes = [axes]

for ax, idx in zip(axes, example_indices):
    sig        = test_signals[idx]
    pulse      = test_pulses[idx]
    pred       = pred_base[idx]
    true_peaks = pulses_to_peaks(pulse, threshold=0.5)
    tp, fp, fn = _match_peaks(pred, true_peaks, tolerance=TOL)

    # Find which true peaks were NOT matched (missed axles)
    matched_true = set()
    matched_pred = set()
    for pi, pp in enumerate(pred):
        dists = np.abs(true_peaks - pp)
        best  = int(np.argmin(dists))
        if dists[best] <= TOL and best not in matched_true:
            matched_true.add(best)
            matched_pred.add(pi)
    missed_peaks   = [true_peaks[j] for j in range(len(true_peaks)) if j not in matched_true]
    spurious_preds = [pred[j] for j in range(len(pred)) if j not in matched_pred]

    samples = np.arange(len(sig))
    ax.plot(samples, sig, color='#444', linewidth=0.8, zorder=1)

    # Ground truth axles
    for tp_pos in [true_peaks[j] for j in matched_true]:
        ax.axvline(tp_pos, color='#2ecc71', alpha=0.8, linewidth=1.2, zorder=2)
    # Missed axles (FN)
    for fn_pos in missed_peaks:
        ax.axvline(fn_pos, color='#e74c3c', alpha=0.9, linewidth=1.8, linestyle='--', zorder=3)
        ax.scatter(fn_pos, sig[fn_pos], color='#e74c3c', s=80, marker='v', zorder=4)
    # Baseline predictions
    for pp in pred:
        ax.scatter(pp, sig[pp], color='steelblue', s=30, marker='|', linewidths=1.5, zorder=3)

    ax.set_xlim(0, len(sig))
    ax.set_ylabel('Strain')
    n_axles = pv.loc[idx, 'n_axles']
    ax.set_title(f'Vehicle #{idx}  |  {n_axles} axles  |  FN={pv.loc[idx,"fn"]}  FP={pv.loc[idx,"fp"]}',
                 fontsize=9)

# Legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color='#444',       lw=1,   label='Signal'),
    Line2D([0], [0], color='#2ecc71',    lw=1.2, label='Detected axle (TP)'),
    Line2D([0], [0], color='#e74c3c',    lw=1.8, linestyle='--', label='Missed axle (FN)'),
    Line2D([0], [0], color='steelblue',  lw=0,   marker='|', markersize=8, label='Baseline prediction'),
]
axes[0].legend(handles=legend_elements, loc='upper right', fontsize=8)
axes[-1].set_xlabel('Sample index')

plt.tight_layout()
plt.show()


## 4. Summary Statistics


In [ ]:
summary = pv.groupby('n_axles').agg(
    total_vehicles  = ('has_fn', 'count'),
    failed_vehicles = ('has_fn', 'sum'),
    total_fn        = ('fn',     'sum'),
    mean_min_spacing= ('min_spacing', 'mean'),
).reset_index()
summary['failure_rate_%'] = (100 * summary['failed_vehicles'] / summary['total_vehicles']).round(1)
summary['mean_min_spacing'] = summary['mean_min_spacing'].round(1)
summary = summary.rename(columns={'n_axles': 'Axle count'})
summary.style.format({'failure_rate_%': '{:.1f}', 'mean_min_spacing': '{:.1f}'}) \
             .bar(subset=['failure_rate_%'], color='#e74c3c', vmin=0, vmax=100)


## Conclusions

The error analysis reveals the root cause of baseline failures:

- **Closely-spaced axles** (tandem/tridem axle groups common on heavy freight trucks) produce overlapping strain pulses that merge into a single broad peak — `find_peaks` detects one peak where there are two or three axles.
- Failure rate increases sharply with axle count: vehicles with more axles have more opportunities for close axle groups.
- The deep learning models (CNN and TCN) overcome this because they learn the *shape* of overlapping pulses from labelled training data, rather than relying on a single amplitude threshold.

This finding supports the thesis argument that a data-driven approach is **necessary** for reliable B-WIM deployment on mixed-traffic roads.
